# 🎓 Project 3 — Student Dropout & Academic Success

## Recruiter-ready project overview

**Problem type:** Multiclass classification  
**UCI dataset:** Predict Students' Dropout and Academic Success (ID 697)

### Executive summary

This project predicts whether a student is likely to be **Dropout, Enrolled, or Graduate** using demographic, academic, socioeconomic and related variables. It demonstrates a practical classification workflow with mixed data, class-level evaluation, feature interpretation and responsible translation of model results into possible intervention ideas.

### What this project demonstrates

- Working with mixed real-world student data
- Data inspection and cleaning
- Categorical and numerical preprocessing
- Multiclass classification
- Decision Tree and Random Forest modeling
- Stratified train/test splitting
- Macro evaluation metrics
- Confusion-matrix interpretation
- Feature importance
- Translating model findings into practical recommendations

### The key data-science question

> **Can available student information help classify a student's academic outcome as Dropout, Enrolled, or Graduate?**

This is a multiclass classification problem because there are three outcome categories.

### Responsible interpretation

A predictive model can identify **patterns associated with outcomes**, but it should not be treated as proof that a particular student will drop out. Real interventions should consider context, fairness, privacy and human judgment.

### Interview takeaway

A strong explanation is: **“This project taught me how to handle a real-world multiclass problem where the useful result is not only the model score, but understanding which groups of variables are associated with different outcomes and how predictions could support earlier intervention.”**


# 🎓 Project 3 — Student Dropout & Academic Success

## Guided Google Colab Machine Learning Project

### Source-based project framing
This project follows the uploaded proposal: the UCI **Predict Students' Dropout and Academic Success** dataset combines academic, demographic, and socio-economic information, with three target classes: **Dropout, Enrolled, and Graduate**. The objective is to identify students at risk of dropping out early enough for intervention and understand important academic and socio-economic predictors.

**Project complexity:** Intermediate — mixed feature types.

### Workflow
1. 🧰 Setup
2. 📥 Load UCI dataset
3. 🔍 Inspect structure and target
4. 🧹 Clean and prepare data
5. 📊 Explore dropout patterns
6. ⚙️ Preprocess mixed feature types
7. ✂️ Create stratified train/test sets
8. 🌳 Train Decision Tree
9. 🌲 Train Random Forest
10. 🔁 Compare models
11. 🧪 Evaluate the selected model
12. 🔲 Analyze the confusion matrix
13. 🔍 Inspect feature importance
14. 🌳 Visualize a shallow Decision Tree
15. 💾 Save outputs
16. 📝 Produce intervention-focused conclusions

> Every explanation is a separate Markdown/text cell. Code cells contain code only.

## 1. 🧰 Setup

### 🔎 What's happening
We install and import the libraries needed to load the UCI data, inspect and visualize it, preprocess mixed feature types, train classification models, and evaluate their predictions.

In [ ]:
!pip -q install ucimlrepo

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)

print("Environment ready!")

## 2. 📥 Load the UCI dataset

### 🔎 What's happening
We retrieve UCI dataset **697**, which is the dataset specified in the project proposal. The package provides the feature table and target table separately.

In [ ]:
students = fetch_ucirepo(id=697)

X = students.data.features.copy()
y = students.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

## 3. 🔍 Inspect the dataset

### 🔎 What's happening
We inspect the columns, data types, missing values, and target labels before deciding how to clean and preprocess the data.

In [ ]:
print("FEATURE INFORMATION")
print("=" * 70)
X.info()

print("\nTARGET COLUMNS")
print(y.columns.tolist())

target_column = y.columns[0]

print("\nTARGET DISTRIBUTION")
display(y[target_column].value_counts())

print("\nMISSING VALUES")
missing = X.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

## 4. 💾 Preserve the original data

### 🔎 What's happening
We create untouched copies before cleaning. This makes the workflow reproducible and gives us a reference to the original UCI data.

In [ ]:
raw_X = X.copy()
raw_y = y.copy()

print("Original feature and target data preserved.")

## 5. 🧹 Clean basic data issues

### 🔎 What's happening
We standardize common missing-value markers and remove exact duplicate records if any exist. We also make sure the target remains aligned with the feature rows.

In [ ]:
X = X.replace(["?", "NA", "N/A", ""], np.nan)

combined = pd.concat(
    [X.reset_index(drop=True), y.reset_index(drop=True)],
    axis=1
)

before = len(combined)
combined = combined.drop_duplicates().reset_index(drop=True)

X = combined[X.columns].copy()
y = combined[[target_column]].copy()

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal :", len(combined))
print("Duplicates removed            :", before - len(combined))

## 6. 📊 Explore the target classes

### 🔎 What's happening
The proposal treats this as a three-class classification problem: **Dropout, Enrolled, and Graduate**. We first examine how many students belong to each class.

In [ ]:
class_counts = y[target_column].value_counts()

display(class_counts.to_frame("Count"))

plt.figure(figsize=(8, 5))
sns.countplot(
    data=y,
    x=target_column,
    order=class_counts.index
)
plt.title("Student Outcome Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

## 7. 👥 Explore demographic dropout patterns

### 🔎 What's happening
The proposal specifically recommends examining dropout trends by demographic factors such as gender and age group. We identify available columns and create plots when those fields are present.

In [ ]:
def find_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    for c in columns:
        lc = str(c).lower()
        if any(candidate.lower() in lc for candidate in candidates):
            return c
    return None

gender_col = find_column(X.columns, ["Gender", "gender"])
age_col = find_column(X.columns, ["Age at enrollment", "Age", "age"])

print("Gender column:", gender_col)
print("Age column:", age_col)

if gender_col:
    temp = pd.concat([X[[gender_col]], y], axis=1)
    rates = pd.crosstab(temp[gender_col], temp[target_column], normalize="index") * 100
    display(rates.round(2))

    rates.plot(kind="bar", figsize=(9, 5))
    plt.title("Student Outcomes by Gender")
    plt.ylabel("Percentage")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

if age_col:
    age_values = pd.to_numeric(X[age_col], errors="coerce")
    age_group = pd.cut(
        age_values,
        bins=[0, 20, 25, 30, 40, 100],
        labels=["≤20", "21–25", "26–30", "31–40", "41+"]
    )

    temp = pd.DataFrame({
        "Age Group": age_group,
        target_column: y[target_column].values
    })

    rates = pd.crosstab(
        temp["Age Group"],
        temp[target_column],
        normalize="index"
    ) * 100

    display(rates.round(2))

    rates.plot(kind="bar", figsize=(10, 5))
    plt.title("Student Outcomes by Age Group")
    plt.ylabel("Percentage")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 8. 💰 Explore scholarship and socio-economic indicators

### 🔎 What's happening
The proposal highlights scholarship status and socio-economic variables as potential intervention signals. We examine available columns that can reveal differences in student outcomes.

In [ ]:
scholarship_col = find_column(
    X.columns,
    ["Scholarship holder", "Scholarship", "scholarship"]
)

print("Scholarship column:", scholarship_col)

if scholarship_col:
    temp = pd.concat([X[[scholarship_col]], y], axis=1)

    rates = pd.crosstab(
        temp[scholarship_col],
        temp[target_column],
        normalize="index"
    ) * 100

    display(rates.round(2))

    rates.plot(kind="bar", figsize=(9, 5))
    plt.title("Student Outcomes by Scholarship Status")
    plt.ylabel("Percentage")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Scholarship column was not identified automatically.")

## 9. 📚 Explore first- and second-semester academic performance

### 🔎 What's happening
The proposal specifically recommends examining first- and second-semester grades and related academic measures. These variables can be especially useful for early-warning interventions.

In [ ]:
grade_candidates = [
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (grade)"
]

grade_columns = [
    c for c in X.columns
    if any(term.lower() in str(c).lower() for term in ["1st sem", "2nd sem"])
    and "grade" in str(c).lower()
]

print("Detected grade columns:")
print(grade_columns)

for col in grade_columns[:4]:
    temp = pd.concat([X[[col]], y], axis=1)
    temp[col] = pd.to_numeric(temp[col], errors="coerce")

    plt.figure(figsize=(9, 5))
    sns.boxplot(data=temp, x=target_column, y=col)
    plt.title(f"{col} by Student Outcome")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 10. 📈 Inspect numerical relationships

### 🔎 What's happening
We identify numerical features and calculate correlations among them. This helps us understand relationships between academic, demographic, and socio-economic variables before modeling.

In [ ]:
numeric_df = X.apply(pd.to_numeric, errors="coerce")
numeric_df = numeric_df.dropna(axis=1, how="all")

print("Number of numerical columns:", numeric_df.shape[1])

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(14, 10))
    sns.heatmap(
        numeric_df.corr(),
        cmap="coolwarm",
        center=0
    )
    plt.title("Correlation Heatmap of Numerical Features")
    plt.tight_layout()
    plt.show()

## 11. ⚙️ Identify feature types

### 🔎 What's happening
This dataset contains mixed feature types. We separate numerical and categorical columns so that each receives an appropriate preprocessing method.

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

## 12. ✂️ Create stratified training and test sets

### 🔎 What's happening
We reserve 20% of the data for final evaluation. Stratification keeps the proportions of Dropout, Enrolled, and Graduate students similar in the training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y[target_column],
    test_size=0.20,
    stratify=y[target_column],
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

## 13. 🧩 Build the preprocessing pipeline

### 🔎 What's happening
Numerical variables are median-imputed and standardized. Categorical variables are filled using the most frequent category and one-hot encoded. The pipeline ensures preprocessing is learned only from the training data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Mixed-type preprocessing pipeline created.")

## 14. 🌳 Define the Decision Tree baseline

### 🔎 What's happening
A Decision Tree is used as the interpretable baseline requested in the proposal. It can learn nonlinear relationships and can later be visualized to explain individual decision rules.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ))
])

print("Decision Tree ready.")

## 15. 🌲 Define the Random Forest model

### 🔎 What's happening
Random Forest combines many decision trees. It is included as the stronger tree-based comparison model and will also provide feature-importance information for interpretation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

print("Random Forest ready.")

## 16. 🔁 Compare models using cross-validation

### 🔎 What's happening
We use 5-fold stratified cross-validation so model performance is not based on only one train/test arrangement. Macro-averaged metrics are emphasized because the proposal asks for fair evaluation across all three classes.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    "Decision Tree": decision_tree,
    "Random Forest": random_forest
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

results = []

for name, model in models.items():
    print(f"Evaluating {name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro Precision": scores["test_precision_macro"].mean(),
        "Macro Recall": scores["test_recall_macro"].mean(),
        "Macro F1": scores["test_f1_macro"].mean()
    })

results_df = pd.DataFrame(results).sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

display(results_df)

## 17. 📊 Visualize model comparison

### 🔎 What's happening
Charts make the differences between the Decision Tree and Random Forest easier to interpret across accuracy, precision, recall, and Macro F1.

In [ ]:
for metric in ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]:
    plt.figure(figsize=(7, 5))
    sns.barplot(
        data=results_df,
        x="Model",
        y=metric
    )
    plt.title(f"Model Comparison — {metric}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 18. 🏆 Select and train the best model

### 🔎 What's happening
We select the model with the highest cross-validated Macro F1 score. Macro F1 balances precision and recall across Dropout, Enrolled, and Graduate.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("Model trained successfully.")

## 19. 🧪 Evaluate final test performance

### 🔎 What's happening
We now test the selected model on data it did not see during training. The classification report shows performance for each student-outcome class.

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("FINAL TEST RESULTS")
print("=" * 70)

print(classification_report(y_test, y_pred))

print("Accuracy        :", round(accuracy_score(y_test, y_pred), 4))
print("Macro Precision :", round(
    precision_score(y_test, y_pred, average="macro", zero_division=0), 4
))
print("Macro Recall    :", round(
    recall_score(y_test, y_pred, average="macro", zero_division=0), 4
))
print("Macro F1        :", round(
    f1_score(y_test, y_pred, average="macro", zero_division=0), 4
))

## 20. 🔲 Analyze the confusion matrix

### 🔎 What's happening
The confusion matrix shows which outcomes are correctly predicted and which classes are confused with each other. This is especially useful for understanding mistakes involving Enrolled and Dropout.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=20)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 21. 🔍 Inspect Random Forest feature importance

### 🔎 What's happening
The Random Forest model can rank transformed features by their contribution to prediction. We use this to identify academic, demographic, and socio-economic variables that deserve attention in the project story.

In [ ]:
random_forest.fit(X_train, y_train)

rf = random_forest.named_steps["model"]
rf_preprocessor = random_forest.named_steps["preprocessor"]

feature_names = rf_preprocessor.get_feature_names_out()

importance = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(25).to_frame("Importance"))

plt.figure(figsize=(10, 9))
importance.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 22. 🌳 Visualize a shallow Decision Tree

### 🔎 What's happening
The proposal recommends visualizing a shallow Decision Tree for storytelling. A shallow tree is easier to read and demonstrates how combinations of features can lead to different student-outcome predictions.

In [ ]:
from sklearn.tree import plot_tree

decision_tree.fit(X_train, y_train)

dt_model = decision_tree.named_steps["model"]
dt_preprocessor = decision_tree.named_steps["preprocessor"]

dt_feature_names = dt_preprocessor.get_feature_names_out()

plt.figure(figsize=(22, 12))

plot_tree(
    dt_model,
    feature_names=dt_feature_names,
    class_names=[str(c) for c in dt_model.classes_],
    filled=True,
    max_depth=3,
    fontsize=8
)

plt.title("Shallow Decision Tree for Interpretability")
plt.show()

## 23. 🎯 Create an early-intervention view

### 🔎 What's happening
For practical storytelling, we calculate how the selected model classifies students into the three outcome groups. This is not a decision to penalize students; it is a way to identify patterns that could support earlier academic or financial assistance.

In [ ]:
intervention_summary = pd.DataFrame({
    "Actual Outcome": y_test.reset_index(drop=True),
    "Predicted Outcome": pd.Series(y_pred)
})

display(
    intervention_summary["Predicted Outcome"]
    .value_counts()
    .rename("Predicted Students")
    .to_frame()
)

print("Potential use:")
print("Use predictions as signals for supportive interventions such as")
print("mentorship, financial-aid outreach, and academic support.")

## 24. 💾 Save all project outputs

### 🔎 What's happening
We save the cleaned dataset, model comparison, predictions, and feature-importance results. These files can be used for a report, portfolio, or further analysis.

In [ ]:
os.makedirs("project3_outputs", exist_ok=True)

X.to_csv(
    "project3_outputs/student_features_cleaned.csv",
    index=False
)

y.to_csv(
    "project3_outputs/student_outcomes.csv",
    index=False
)

results_df.to_csv(
    "project3_outputs/model_comparison.csv",
    index=False
)

importance.to_csv(
    "project3_outputs/random_forest_feature_importance.csv"
)

intervention_summary.to_csv(
    "project3_outputs/test_predictions.csv",
    index=False
)

print("Project outputs saved successfully.")

## 25. 📝 Final project summary

### 🔎 What's happening
This section records the key results that should appear in the final project report. Replace the automatically generated values with your interpretation after running the notebook.

In [ ]:
best_row = results_df.iloc[0]

print("=" * 75)
print("PROJECT 3 — STUDENT DROPOUT & ACADEMIC SUCCESS")
print("=" * 75)

print("\nBest model:", best_model_name)
print(f"Cross-validated Accuracy        : {best_row['Accuracy']:.4f}")
print(f"Cross-validated Macro Precision : {best_row['Macro Precision']:.4f}")
print(f"Cross-validated Macro Recall    : {best_row['Macro Recall']:.4f}")
print(f"Cross-validated Macro F1        : {best_row['Macro F1']:.4f}")

print("\nFinal Test Accuracy:")
print(f"{accuracy_score(y_test, y_pred):.4f}")

print("\nTop Random Forest features:")
for feature in importance.head(10).index:
    print("-", feature)

print("\nProject completed successfully.")

# 🏁 Project 3 Conclusion Template

### Problem
The project aimed to predict whether a student would be **Dropout, Enrolled, or Graduate**, while identifying academic and socio-economic factors associated with student outcomes.

### Results
The best-performing model was **[MODEL]**, achieving a cross-validated Macro F1 of **[VALUE]** and a final test Macro F1 of **[VALUE]**.

### Important predictors
The strongest model features included **[FEATURES]**. These should be interpreted as predictive associations rather than proof that a feature causes dropout.

### Recommended intervention direction
The project proposal suggests supportive interventions such as:

- Early mentorship programs
- Financial-aid outreach
- Academic support
- Monitoring early academic performance

### Limitation
Model predictions should be treated as **early-warning signals**, not as automatic decisions about a student's future. Any real institutional use should include human review, fairness checks, privacy protections, and appropriate student support.

## 🎯 Portfolio conclusion — Project 3

This project demonstrates how machine learning can be used to study student outcomes while keeping interpretation responsible.

### Knowledge checkpoints

- **Target:** the outcome the model predicts.
- **Feature:** an input variable used to make the prediction.
- **Stratification:** preserve class proportions during splitting.
- **Confusion matrix:** shows which classes are being confused.
- **Feature importance:** indicates which inputs contributed most to a tree-based model.

### What to remember

Prediction is not causation. A useful model can reveal patterns without proving that changing one variable will cause a student's outcome to change.
